In [1]:
# ==============================================================================
# NOTEBOOK 3: ANALYSE EXPLORATOIRE DES MÉTADONNÉES (EDA)
# PROJET: Text Mining - L'Enquêteur du Niger
# ==============================================================================

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Configuration esthétique des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10

# 1. Chargement des données prétraitées
INPUT_PATH = "../data/processed/enqueteur_preprocessed.csv"
FIGURES_DIR = "../outputs/figures/"
os.makedirs(FIGURES_DIR, exist_ok=True)

df = pd.read_csv(INPUT_PATH)
df['date_utc'] = pd.to_datetime(df['date_utc'])
print(f"Dataset chargé pour l'EDA : {df.shape[0]} articles et {df.shape[1]} colonnes.")


Dataset chargé pour l'EDA : 339 articles et 19 colonnes.


In [2]:
# ==============================================================================
# ÉTAPE 1 : ANALYSE DU RYTHME DES PUBLICATIONS (ÉVOLUTION CHRONOLOGIQUE)
# ==============================================================================
print("\n--- Analyse de l'activité éditoriale ---")

# Groupement par mois pour voir l'évolution de la production d'articles
publications_par_mois = df.groupby(df['date_utc'].dt.to_period('M')).size()

plt.figure()
publications_par_mois.plot(kind='bar', color='skyblue', edgecolor='black')
plt.title("Nombre d'articles publiés par mois (L'Enquêteur)", fontsize=14, fontweight='bold')
plt.xlabel("Période", fontsize=12)
plt.ylabel("Nombre de posts", fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "rythme_publications.png"))
plt.close()

print(f"Moyenne d'articles publiés par mois : {publications_par_mois.mean():.1f}")
print(f"Mois le plus actif : {publications_par_mois.idxmax()} ({publications_par_mois.max()} articles)")



--- Analyse de l'activité éditoriale ---


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13580\2141985764.py:7: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  publications_par_mois = df.groupby(df['date_utc'].dt.to_period('M')).size()


Moyenne d'articles publiés par mois : 22.6
Mois le plus actif : 2025-07 (41 articles)


In [3]:
# ==============================================================================
# ÉTAPE 2 : STATISTIQUES DESCRIPTIVES DE L'ENGAGEMENT
# ==============================================================================
print("\n--- Statistiques descriptives de l'engagement global ---")
metriques_engagement = ['nb_commentaires', 'nb_partages', 'like', 'colere', 'triste', 'love']

# Résumé statistique (Moyenne, Médiane, Max, Somme totale)
stats_desc = df[metriques_engagement].describe().loc[['mean', '50%', 'max']]
stats_desc.loc['total_cumule'] = df[metriques_engagement].sum()
print(stats_desc.round(1))



--- Statistiques descriptives de l'engagement global ---
              nb_commentaires  nb_partages     like  colere  triste    love
mean                     34.7         20.0    209.9     0.2     1.5     4.6
50%                      22.0         12.0    171.0     0.0     0.0     2.0
max                     494.0        214.0   1551.0     8.0    85.0   152.0
total_cumule          11762.0       6770.0  71163.0    51.0   524.0  1547.0


In [4]:
# ==============================================================================
# ÉTAPE 3 : ANALYSE DES RÉACTIONS DE COLÈRE (L'INDIGNATION DU PUBLIC)
# ==============================================================================
# La colère (Angry Count) est une métrique cruciale pour mesurer l'impact des dénonciations du journal.
df['part_colere'] = df['colere'] / (df['like'] + df['colere'] + df['love'] + df['triste'] + 1e-5)

# Top 5 des articles ayant suscité la plus forte proportion de colère (avec au moins 10 réactions au total)
df_reactions_min = df[(df['like'] + df['colere'] + df['love']) > 10]
top_colere = df_reactions_min.sort_values(by='part_colere', ascending=False).head(5)

print("\n--- Top 5 des articles ayant provoqué la plus forte indignation (% Colère) ---")
for idx, row in top_colere.iterrows():
    print(f"\n[Date: {row['date_utc'].strftime('%Y-%m-%d')}] | Part de colère : {row['part_colere']*100:.1f}%")
    print(f"Extrait : {row['texte'][:150]}...")
    print(f"Lien : {row['url_post']}")



--- Top 5 des articles ayant provoqué la plus forte indignation (% Colère) ---

[Date: 2025-06-17] | Part de colère : 2.8%
Extrait : Le gambit du Général Tiani : Céder des pions pour mieux gagner la partie
(Editorial de Soumana I. Maïga, Quotidien L’Enquêteur du mardi 17 Juin 2025)
...
Lien : https://www.facebook.com/permalink.php?story_fbid=pfbid02gd9MuD23smkLMgxFxLezMBxPfBw6TnL989ZNMAuHRoWrky3epbLLBgu7715XSApEl&id=100063543878289

[Date: 2025-06-27] | Part de colère : 2.1%
Extrait : MAUVAISE QUALITÉ DES SERVICES TÉLÉCOMS AU NIGER 
Le théâtre de l'impuissance et des piteuses excuses des opérateurs
(Quotidien L’Enquêteur du Vendredi...
Lien : https://www.facebook.com/permalink.php?story_fbid=pfbid02G3Bv4D5qzdKGh2H2HtTFk4pXq1Z9yuvWMUk9zKed2GUvqXRLAW9VdDjunVuegkNil&id=100063543878289

[Date: 2025-11-05] | Part de colère : 1.8%
Extrait : 2 Milliards $ vendus, 66 Millions $ touchés par le Niger : l'arnaque du siècle de CNPC ?
(Editorial de Soumana I. Maïga, Quotidien L’Enquêteur du Merc..

In [5]:
# ==============================================================================
# ÉTAPE 4 : CORRÉLATION ENTRE LES DIFFÉRENTES RÉACTIONS
# ==============================================================================
plt.figure(figsize=(8, 6))
correlation_matrix = df[metriques_engagement].corr()
sns.heatmap(correlation_matrix, annot=True, cmap="coolwarm", fmt=".2f", square=True)
plt.title("Corrélations entre les différentes formes d'engagement", fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "correlation_engagement.png"))
plt.close()

print("\nVisualisations sauvegardées dans le dossier 'outputs/figures/' !")


Visualisations sauvegardées dans le dossier 'outputs/figures/' !
